# 06 TrainTestSplit

Notebook นี้ใช้สำหรับแบ่งข้อมูลที่ผ่านการ clean และ feature engineering แล้ว
ออกเป็น Train set และ Test set เพื่อใช้ในการ train และ evaluate model


In [11]:
import pandas as pd
import numpy as np

TEST_SIZE = 0.2  # 20% test (ช่วงเวลาล่าสุด), 80% train (ช่วงเวลาก่อนหน้า)


## Load Clean Data

โหลดข้อมูลที่ผ่านการลบ outlier แล้วจาก `05_OutlierHandling.ipynb`


In [12]:
source_path = '../../data/interim/vw_timestamp_dashboard_featured_no_outlier.csv'
df = pd.read_csv(source_path, encoding='utf-8-sig')

print(f'Source : {source_path}')
print(f'Shape  : {df.shape}')
df.head()


Source : ../../data/interim/vw_timestamp_dashboard_featured_no_outlier.csv
Shape  : (22085, 68)


,PlantName,PickListType,PickDate,TruckSeqNo,CarType,CarNo,PackListNo,CustomerName,QueueTime,PrepareForward,...,total_queue,available_bays,sap_per_group,queue_x_bays,queue_x_sap,rolling_avg_time_last5,inter_arrival_min,car_type_x_sap,avg_time_by_cartype,rolling_avg_cartype_last10
0,SB1,0.0,2025-01-02 07:21:43,2.0,3.0,71-3545,SB1PL250102002,"SCGR Cholburi Plant SCG Roofing Co., Ltd.",2025-01-02 07:21:46,0.0,...,0,10,6720.0,0.0,0.0,72.05,2.80,20160.0,74.15,72.05
1,SB1,0.0,2025-01-02 10:52:33,3.0,1.0,89-1359,SB1PL250102004,หจก.สำรวยเซรามิค,2025-01-02 10:52:39,0.0,...,0,10,809.0,0.0,0.0,77.01,210.85,1618.0,52.28,52.28
2,SB1,0.0,2025-01-02 12:05:56,4.0,3.0,71-6663,SB1PL250102009,"SCGR Lumpoon Plant SCG Roofing Co., Ltd.",2025-01-02 12:05:59,0.0,...,0,10,523.0,0.0,0.0,64.80,73.35,1569.0,74.15,77.01
3,SB1,0.0,2025-01-02 13:12:26,5.0,1.0,70-6399,SB1PL250102010,กรุงเทพฯ ดอนเมือง,2025-01-02 13:12:30,0.0,...,0,10,1300.0,0.0,0.0,58.10,66.50,1300.0,52.28,40.38
4,SB1,0.0,2025-01-02 13:13:05,6.0,1.0,73-0454,SB1PL250102011,บ.อุดมชัยสมุทรสงครามซิเมนต์ จก.,2025-01-02 13:13:12,0.0,...,1,10,797.5,0.1,1595.0,52.21,0.70,1595.0,52.28,34.53


## Define Features and Target

กำหนด Feature columns (`X`) และ Target column (`y`)

- **Target** : `loading_time_min`
- **Drop columns** : คอลัมน์ที่เป็น target time อื่น ๆ และ ID/datetime columns ที่ไม่ใช้ใน model


In [13]:
from sklearn.preprocessing import OrdinalEncoder

TARGET = 'total_time_min'

DROP_COLS = [
    # 1) Leakage — sub-time targets
    'wait_call_min', 'prepare_loading_min', 'close_job_min', 'loading_time_min',

    # 2) ID / raw datetime
    'PackListNo', 'CarNo', 'CustomerName',
    'PickDate', 'QueueTime', 'OperatorCarConfirm', 'CarConfirm',
    'FirstPostPallet', 'LastPostPallet', 'PostingTime',
    'TileStart', 'TileEnd', 'FittingStart', 'FittingEnd', 'AccStart', 'AccEnd',
    'TruckReceiveDate', 'CreateDate', 'PickingTime',
    'date_key',

    # 3) Useless — 1 unique value
    'PlantName', 'TruckStatus', 'PackListStatus',

    # 4) Low signal / replaced
    'is_weekend', 'product_mix_ratio',
    'queue_not_posted',

    # 5) Low importance / redundant
    'CPACTileSapAmount', 'PRESTIGETileSapAmount', 'CPACFittingSapAmount',
    'PRESTIGEFittingSapAmount', 'ACCESSORIESSapAmount',
    'NEUSTILETileSapAmount', 'NEUSTILEFittingSapAmount', 'DURAFittingSapAmount',
    'is_overload_proxy', 'TruckReceiveMinute', 'has_tile', 'has_fitting',
    'has_accessories', 'TruckReceiveHour', 'avg_loading_rate_by_car',
    'prepare_overhead_interaction', 'is_early_truck', 'concurrent_trucks_loading',
    'year', 'time_of_day', 'workload_x_time_of_day',

    # # 6) Near-zero correlation (r < 0.05) — ไม่มี signal จริง
    # 'day_of_week',       # r=-0.001
    # 'inter_arrival_min', # r=-0.005
    # 'hour',              # r=+0.018
    # 'week_of_month',     # r=+0.020
    # 'month',             # r=-0.025
    # 'PostLocationName',  # r=+0.037
    # 'TruckSeqNo',        # r=+0.041  (sequential ID ไม่ใช่ signal)

    # # 7) Collinear / redundant กับ feature อื่น
    # 'total_accessories_amount', # r=+0.093  ต่ำ, minor part of total_sap_amount
    # 'queue_closing',            # r=+0.011  แทบเป็นศูนย์
    # 'total_queue',              # = queue_waiting + queue_loading + queue_closing
    # 'available_bays',           # = 10 - queue_loading (perfect collinear)
    # 'queue_x_bays',             # r=+0.057  interaction ของ signal อ่อน
    # 'avg_time_by_cartype',      # r=+0.524  แทนด้วย rolling_avg_cartype_last10 ที่ไม่มี leakage

    TARGET,
]

drop_existing = [c for c in DROP_COLS if c in df.columns]
X = df.drop(columns=drop_existing)
y = df[TARGET]

# encode categorical เพื่อคำนวณ correlation เท่านั้น (X เก็บ string ไว้ให้ tree model)
cat_cols = X.select_dtypes(include='object').columns.tolist()
X_encoded = X.copy()
if cat_cols:
    enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X_encoded[cat_cols] = enc.fit_transform(X[cat_cols].astype(str))

print(f'X shape  : {X.shape}')
print(f'y shape  : {y.shape}')
print(f'\nFeatures ({len(X.columns)}):')
for i, c in enumerate(X.columns, 1):
    r = X_encoded[c].corr(y)
    tag = ' [cat]' if c in cat_cols else ''
    print(f'  {i:2d}. {c:<45}  r={r:+.3f}{tag}')


X shape  : (22085, 27)
y shape  : (22085,)

Features (27):
   1. PickListType                                   r=+0.276
   2. TruckSeqNo                                     r=+0.041
   3. CarType                                        r=+0.524
   4. PrepareForward                                 r=+0.294
   5. PostLocationName                               r=+0.037
   6. hour                                           r=+0.018
   7. day_of_week                                    r=-0.001
   8. week_of_month                                  r=+0.020
   9. month                                          r=-0.025
  10. total_tile_amount                              r=+0.581
  11. total_fitting_amount                           r=+0.332
  12. total_accessories_amount                       r=+0.093
  13. total_sap_amount                               r=+0.614
  14. product_group_count                            r=+0.228
  15. queue_waiting                                  r=+0.051
  16. queue

## Temporal Train-Test Split

แบ่งข้อมูลตาม **เวลา** เพื่อป้องกัน data leakage

- เรียงข้อมูลตาม `OperatorCarConfirm` (เวลารถเข้าโรงงาน) จากเก่าไปใหม่
- **Train** : 80% แรก (ข้อมูลเก่า)
- **Test**  : 20% หลัง (ข้อมูลใหม่ที่ model ไม่เคยเห็น)

> Random split จะทำให้ข้อมูลในอนาคตรั่วเข้า train set ได้ ทำให้ผล evaluation ดีเกินจริง

In [14]:
# เรียงตาม OperatorCarConfirm (เวลารถเข้าโรงงาน)
date_series = pd.to_datetime(df['OperatorCarConfirm'])
sorted_idx = date_series.sort_values().index

X = X.loc[sorted_idx].reset_index(drop=True)
y = y.loc[sorted_idx].reset_index(drop=True)
dates = date_series.loc[sorted_idx].reset_index(drop=True)

# หา cutoff index และ cutoff date
cutoff = int(len(X) * (1 - TEST_SIZE))
cutoff_date = dates.iloc[cutoff]

X_train, X_test = X.iloc[:cutoff].copy(), X.iloc[cutoff:].copy()
y_train, y_test = y.iloc[:cutoff].copy(), y.iloc[cutoff:].copy()

print(f'Cutoff date  : {cutoff_date}')
print()
print(f'Train : X={X_train.shape}  y={y_train.shape}')
print(f'  date range : {dates.iloc[0].date()} → {dates.iloc[cutoff - 1].date()}')
print()
print(f'Test  : X={X_test.shape}   y={y_test.shape}')
print(f'  date range : {dates.iloc[cutoff].date()} → {dates.iloc[-1].date()}')
print()
print(f'Train target stats:')
print(y_train.describe().round(2))
print()
print(f'Test  target stats:')
print(y_test.describe().round(2))


Cutoff date  : 2026-02-14 09:39:14

Train : X=(17668, 27)  y=(17668,)
  date range : 2025-01-02 → 2026-02-14

Test  : X=(4417, 27)   y=(4417,)
  date range : 2026-02-14 → 2026-05-18

Train target stats:
count    17668.00
mean        54.39
std         21.53
min          5.43
25%         38.49
50%         51.50
75%         67.43
max        120.00
Name: total_time_min, dtype: float64

Test  target stats:
count    4417.00
mean       55.57
std        21.69
min         7.47
25%        39.03
50%        52.85
75%        69.63
max       119.58
Name: total_time_min, dtype: float64


## Save Train / Test Data

บันทึกไฟล์ทั้ง Train และ Test ไปยังโฟลเดอร์ `data/processed`


In [15]:
from pathlib import Path

train_df = X_train.copy()
train_df[TARGET] = y_train.values

test_df = X_test.copy()
test_df[TARGET] = y_test.values

output_dir = Path('../../data/processed')
output_dir.mkdir(parents=True, exist_ok=True)

train_path = output_dir / 'train.csv'
test_path  = output_dir / 'test.csv'

train_df.to_csv(train_path, index=False, encoding='utf-8-sig')
test_df.to_csv(test_path,  index=False, encoding='utf-8-sig')

print(f'Train saved : {train_path}  ({len(train_df)} rows, {len(train_df.columns)} cols)')
print(f'Test  saved : {test_path}   ({len(test_df)} rows, {len(test_df.columns)} cols)')
print(f'\nAll features in train.csv ({len(train_df.columns) - 1} features + target):')
for i, c in enumerate([c for c in train_df.columns if c != TARGET], 1):
    print(f'  {i:2d}. {c}')

Train saved : ..\..\data\processed\train.csv  (17668 rows, 28 cols)
Test  saved : ..\..\data\processed\test.csv   (4417 rows, 28 cols)

All features in train.csv (27 features + target):
   1. PickListType
   2. TruckSeqNo
   3. CarType
   4. PrepareForward
   5. PostLocationName
   6. hour
   7. day_of_week
   8. week_of_month
   9. month
  10. total_tile_amount
  11. total_fitting_amount
  12. total_accessories_amount
  13. total_sap_amount
  14. product_group_count
  15. queue_waiting
  16. queue_loading
  17. queue_closing
  18. total_queue
  19. available_bays
  20. sap_per_group
  21. queue_x_bays
  22. queue_x_sap
  23. rolling_avg_time_last5
  24. inter_arrival_min
  25. car_type_x_sap
  26. avg_time_by_cartype
  27. rolling_avg_cartype_last10
